# Data

In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt -n 3

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [4]:
import collections
import numpy as np
import tqdm

def load(limit, path = "stand/log.local.txt"):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    with open(path) as f:
        req = list()
        reqid = None
        models = list()
        prevreqmodel = None
        reqmodel = dict()
        prevmodelid = -1
        bannermodelid = -1
        for i, line in tqdm.tqdm_notebook(enumerate(f)):
            if line.startswith("Model = 6;"):
                prevreqmodel = reqmodel
                reqmodel = dict()

            if line.startswith("Model = "):
                spl = line.split(" ")
                prevmodelid = int(spl[2][:-1])
                bannermodelid = max(bannermodelid , prevmodelid)
                reqmodel[prevmodelid] = readvector(spl[3])
            elif line.startswith("dbid"):
                spl = line.split(" ")
                dbid = int(spl[1][:-1])
                docembs[bannermodelid][dbid] = readvector(spl[2])
            elif line.startswith("seed"):
                if len(requests) >= limit:
                    break
                if req:
                    requests.append((reqid, prevreqmodel, sorted(req)))
                    req = list()
                reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
            else:
                req.append(
                    (int(line.split()[0]), float(line.split()[1]))
                )

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7):
            self.games_count = games_count
            self.key_games = np.random.choice(np.arange(games_count), key_size, replace=False)

            self.requests = requests
            np.random.shuffle(self.requests)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext()

In [5]:
ctx = load(1500)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
101 937 446


# Models

In [6]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in self.ctx.get_requests(t)
        ])
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):
        mses = list()
        results = list()
        for rec, tru in zip(self.recommend(t), self.get_user_scores(t)):
            mses.append((rec - tru) ** 2)
            rec = np.argsort(-rec)
            tru = np.argsort(-tru)
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            results.append(ev(rec[:topsize], tru[:topsize]) / float(topsize))
        print(np.mean(results), np.array(mses).mean(), len(results))
        return np.mean(results)

array([9, 4])

In [19]:
dir(tfr.keras.losses)

['ApproxMRRLoss',
 'ApproxNDCGLoss',
 'ClickEMLoss',
 'DCGLambdaWeight',
 'GumbelApproxNDCGLoss',
 'ListMLELambdaWeight',
 'ListMLELoss',
 'MeanSquaredLoss',
 'NDCGLambdaWeight',
 'PairwiseHingeLoss',
 'PairwiseLogisticLoss',
 'PairwiseSoftZeroOneLoss',
 'PrecisionLambdaWeight',
 'RankingLossKey',
 'SigmoidCrossEntropyLoss',
 'SoftmaxLoss',
 'UniqueSoftmaxLoss',
 '_ListwiseLoss',
 '_PairwiseLoss',
 '_RankingLoss',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'get',
 'losses_impl',
 'tf',
 'utils']

In [141]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "train_popbias": False,
    "train_bias": False,
    "verbose": True,
    "train_dssm": False,
    "loss": "mse"
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        self.fit_kwargs = fit_kwargs
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[2][i][1] for i in ctx.key_games])
                    for r_i in ctx.get_requests(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[2][g_i][1] for r in ctx.get_requests("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in ctx.get_requests("key")
        ])
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return self.embed_users[t]
    
    def get_game_embs(self):
        return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.train_bias = p["train_bias"]
        self.train_popbias = p["train_popbias"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        
        for i in range(p["n"]):
            def loss():
                def get_logits():
                    if self.train_dssm:
                        logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs))
                    else:
                        logits = train_user_embs @ self.W @ game_embs.T
                    
                    if self.train_popbias:
                        logits += self.pb * self.game_avg_scores["train"]     
                        
                    if self.train_bias:
                        logits += self.b

                    return logits
                        
                if self.loss == "mse":
                    logits = get_logits()
                    v = tf.reduce_mean((train_user_scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits = get_logits()
                    q_mean = train_user_scores.mean(axis=1)
                    # print(train_user_scores.shape, q_mean.shape)
                    v = tf.reduce_mean(((train_user_scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    loss_batch = 64 if "loss_batch" not in p else p["loss_batch"]
                    while True:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        if self.train_dssm:
                            logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs[game_slice]))
                        else:
                            logits = train_user_embs @ self.W @ game_embs[game_slice].T

                        if self.train_popbias:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                        if self.train_bias:
                            logits += self.b

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            train_user_scores[:, game_slice].astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    loss_batch = 64 if "loss_batch" not in p else p["loss_batch"]
                    game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                    if self.train_dssm:
                        logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs[game_slice]))
                    else:
                        logits = train_user_embs @ self.W @ game_embs[game_slice].T

                    if self.train_popbias:
                        logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_bias:
                        logits += self.b

                    scores = train_user_scores[:, game_slice]
                    # v = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
                    #    (scores.T > np.quantile(scores, 1 - 100./16000).T).T,
                    #    logits
                    #))
                    v = -tf.reduce_mean(tf.where(
                        (scores.T > np.quantile(scores, p["loss_q"]).T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                else:
                    assert False
                    
                v += tf.reduce_mean(self.W * self.W) * p["c"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            else:
                weights += [self.W]
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_bias:
                weights += [self.b]   
                
                
            opt.minimize(loss, weights).numpy()

    def recommend(self, t):
        if self.train_dssm:
            logits = self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))
        else:
            logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [ ]:
logs = list()

ev([Popular(ctx)], logs)

for n in [1, 10, 100, 1000, 10000, 20000, 50000, 100000]:
    for train_dssm in [False, True]:
        c_grid = [0.1] if train_dssm else [0, 0.01, 0.1, 1, 10, 100, 1000]
        for c in c_grid:
            for train_popbias in [False, True]:
                for train_bias in [False, True]:
                    for loss in ["mse", "qmse", "ApproxNDCGLoss", "softmax"]:
                        lb_grid = [8, 16, 32, 64, 128] if loss in ["ApproxNDCGLoss", "softmax"] else [32]
                        for lb in lb_grid:
                            kw = {
                                "c": c,
                                "train_dssm": train_dssm,
                                "train_popbias": train_popbias,
                                "train_bias": train_bias,
                                "verbose": False,
                                "loss": loss,
                                "loss_batch": lb,
                                "loss_q": (lb - 1.5) / lb,
                                "n": n
                            }
                            ev([
                                DssmKnn(ctx, 6, fit_kwargs=kw),
                            ], logs)
                            ev([
                                CBKnnV0(ctx, fit_kwargs=kw),
                            ], logs)




model =  <__main__.Popular object at 0x7f40703ddb00>
0.4506830309498399 1.4365100438225953 937
0.4770627802690583 1.3799666957303232 446
0.4506830309498399 0.4770627802690583
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1})
0.0003521878335112059 1.487868 937
0.00020179372197309415 1.4288288 446
0.0003521878335112059 0.00020179372197309415
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': Fal

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 8, 'loss_q': 0.8125, 'n': 1})
0.4449413020277482 3.6134121 937
0.4709641255605381 4.225232 446
0.4449413020277482 0.4709641255605381
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 1})
0.02690501600853789 1.4895349 937
0.026614349775784755 1.43

0.027737459978655284 1.4901778 937
0.027735426008968608 1.430872 446
0.027737459978655284 0.027735426008968608
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 1})
0.439871931696905 3.3865557 937
0.4647982062780269 3.9684284 446
0.439871931696905 0.4647982062780269
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss':

self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 1})
0.07700106723585913 1.4897773 937
0.08161434977578474 1.4304956 446
0.07700106723585913 0.08161434977578474
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 1})
0.44611526147278546 3.2509356 937
0.4723991031390135 3.7665374 446
0.44611526147278546 0.472

0.46363927427961577 3.2463384 937
0.48912556053811657 3.7102246 446
0.46363927427961577 0.48912556053811657
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 8, 'loss_q': 0.8125, 'n': 1})
0.44998932764140875 1.4364861 937
0.47623318385650226 1.3799752 446
0.44998932764140875 0.47623318385650226
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'soft

...

0.5162860192102454 4.096569 937
0.5340807174887893 4.151371 446
0.5162860192102454 0.5340807174887893
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1000})
0.5535218783351121 257944.95 937
0.5730717488789239 330006.0 446
0.5535218783351121 0.5730717488789239
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'Ap

self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.3862113127001067 1.411107 937
0.39082959641255605 1.3612909 446
0.3862113127001067 0.39082959641255605
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.6337566702241195 1.2860627 937
0.6418161434977578 1.3126186 446
0.6337566702241195 0.641816143497

0.3882283884738527 149.36917 937
0.39560538116591926 154.59811 446
0.3882283884738527 0.39560538116591926
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 10000})
0.5650907150480257 134994.6 937
0.5888565022421526 171794.73 446
0.5650907150480257 0.5888565022421526
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'sof

self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.429615795090715 17.912437 937
0.4332511210762332 18.655596 446
0.429615795090715 0.4332511210762332
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5918783351120599 27361.273 937
0.6110986547085202 33543.94 446
0.591878335112059

0.6373319103521877 4201.0957 937
0.6554035874439462 4641.989 446
0.6373319103521877 0.6554035874439462
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5427107790821771 1.3815161 937
0.5561210762331839 1.3328975 446
0.5427107790821771 0.5561210762331839
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'los

In [143]:
logs

[('<__main__.Popular object at 0x7f40703ddb00>',
  0.4506830309498399,
  0.4770627802690583),
 ("DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1})",
  0.0003521878335112059,
  0.00020179372197309415),
 ("CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1})",
  0.43829242262540025,
  0.46473094170403584),
 ("DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'qmse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1})",
  0.2618996798292423,
  0.2662780269058296),
 ("CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'qmse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1})",
  0.43508004268943434,
  0.46),
 ("DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbia

In [144]:
sorted(logs, key = lambda x : -x[2])

[("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 10000})",
  0.6835965848452508,
  0.6814573991031391),
 ("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 10000})",
  0.6814941302027748,
  0.6798878923766817),
 ("CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.6527854855923159,
  0.6694394618834082),
 ("CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.6523906083244396,
  0.668811659192825),
 ("CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss':

In [145]:
logs[-1]

("CbKnn({'c': 0.01, 'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 8, 'loss_q': 0.8125, 'n': 20000})",
 0.5532977588046959,
 0.5743946188340807)

In [ ]:
for n in [20000, 50000, 100000]:
    for train_dssm in [False, True]:
        c_grid = [0.1] if train_dssm else [0, 0.01, 0.1, 1, 10, 100, 1000]
        for c in c_grid:
            for train_popbias in [False, True]:
                for train_bias in [False, True]:
                    for loss in ["mse", "qmse", "ApproxNDCGLoss", "softmax"]:
                        lb_grid = [8, 16, 32, 64, 128] if loss in ["ApproxNDCGLoss", "softmax"] else [32]
                        for lb in lb_grid:
                            kw = {
                                "c": c,
                                "train_dssm": train_dssm,
                                "train_popbias": train_popbias,
                                "train_bias": train_bias,
                                "verbose": False,
                                "loss": loss,
                                "loss_batch": lb,
                                "loss_q": (lb - 1.5) / lb,
                                "n": n
                            }
                            ev([
                                DssmKnn(ctx, 6, fit_kwargs=kw),
                            ], logs)
                            ev([
                                CBKnnV0(ctx, fit_kwargs=kw),
                            ], logs)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 20000})
0.3839381003201708 1.4075162 937
0.38585201793721974 1.3587918 446
0.3839381003201708 0.38585201793721974
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 20000})
0.6328601921024547 1.2845656 937
0.6399551569506725 

# EOLN

In [118]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 16,
        "loss_q": 14./16,
        "n": 1000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 16, 'loss_q': 0.875, 'n': 1000})
0.34623265741728926 446.67618 937
0.36152466367713004 454.78464 446
0.34623265741728926 0.36152466367713004


In [119]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 16,
        "loss_q": 8./16,
        "n": 1000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 16, 'loss_q': 0.5, 'n': 1000})
0.2629882604055496 3076.6821 937
0.27887892376681617 3168.362 446
0.2629882604055496 0.27887892376681617


In [121]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 16,
        "loss_q": 14.5/16,
        "n": 1000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 1000})
0.34935965848452505 735.0801 937
0.3630269058295964 754.2264 446
0.34935965848452505 0.3630269058295964


In [123]:
100./16000 * 32

0.2

In [124]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 32,
        "loss_q": 30.5/32,
        "n": 1000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 1000})
0.39674493062966915 205.35905 937
0.41089686098654715 204.78824 446
0.39674493062966915 0.41089686098654715


In [125]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 32,
        "loss_q": 30.5/32,
        "n": 10000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5623052294557098 161.11317 937
0.5743273542600896 152.08694 446
0.5623052294557098 0.5743273542600896


In [126]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "c": 0.1,
        "train_dssm": True,
        "train_popbias": True,
        "verbose": False,
        "loss": "softmax",
        "loss_batch": 32,
        "loss_q": 30.5/32,
        "n": 20000
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 20000})
0.5995624332977587 71.21089 937
0.6083183856502243 70.35767 446
0.5995624332977587 0.6083183856502243


In [55]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": True,
        "train_popbias": True,
        "verbose": True,
        "loss": "ApproxNDCGLoss",
        "loss_batch": 128
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': True, 'train_popbias': True, 'verbose': True, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-7.5887147e-05, -1.2828424e-04, -6.0376676e-04, ...,
         5.6712003e-04,  2.5928780e-04, -3.5279826e-04],
       [ 5.4057583e-04, -1.5171326e-03,  1.6102612e-04, ...,
        -1.3803825e-03, -6.6707136e-05, -1.3386350e-03],
       [-8.0792082e-04,  1.0303992e-03, -2.0434991e-04, ...,
         8.5581181e-04,  1.5884810e-03, -4.9285084e-04],
       ...,
       [ 1.4668151e-03, -1.2742650e-03, -1.6374618e-04, ...,
        -1.5732030e-03, -1.3884255e-03,  1.6074238e-03],
       [-1.3060219e-03,  1.5595848e-03, -1.3225897e-03, ...,
        -

-0.8208884
-0.86225456
-0.8595515
-0.70639694
nanloss ignored
nanloss ignored
nanloss ignored
-0.8608353
nanloss ignored
-0.7857102
nanloss ignored
nanloss ignored
-0.89800006
nanloss ignored
nanloss ignored
-0.8827596
nanloss ignored
-0.878774
-0.8657405
nanloss ignored
nanloss ignored
nanloss ignored
-0.7846873
-0.85299593
nanloss ignored
nanloss ignored
-0.9131562
-0.8833353
-0.8581872
-0.8697098
-0.85620886
-0.87362325
nanloss ignored
-0.7863883
-0.78553534
-0.86461025
-0.6914747
nanloss ignored
-0.8512509
nanloss ignored
-0.7555313
-0.64564896
-0.8393349
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.9115392
-0.83535683
-0.81371033
-0.79839
nanloss ignored
nanloss ignored
-0.8783312
-0.8459198
-0.8691107
-0.83637506
nanloss ignored
nanloss ignored
-0.83487934
nanloss ignored
nanloss ignored
-0.8504423
-0.85025716
nanloss ignored
nanloss ignored
nanloss ignored
-0.89216906
nanloss ignored
-0.8471448
nanloss ignored
nanloss ignored
nanloss ignored

In [56]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "verbose": True,
        "loss": "ApproxNDCGLoss",
        "loss_batch": 128
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'verbose': True, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 6.8665715e-04, -1.3647830e-03, -1.0317456e-03, ...,
         7.6511828e-04,  1.7247116e-03,  1.2842363e-03],
       [ 1.4839476e-03,  1.5405220e-03,  1.0651678e-03, ...,
        -8.2541403e-05,  7.0944143e-04,  1.6581819e-03],
       [-5.9576332e-06,  8.8483247e-04, -1.5227513e-03, ...,
         2.0681098e-04, -1.3601002e-03,  2.5168582e-04],
       ...,
       [-1.4007863e-03,  1.1457499e-03,  1.5745398e-03, ...,
         2.3704619e-04,  4.1447819e-04, -1.2046102e-03],
       [ 1.3699639e-03,  1.4124603e-04,  2.1316469e-04, ...,
        

nanloss ignored
nanloss ignored
-0.668466
-0.7781037
-0.62016636
-0.82981086
-0.749465
nanloss ignored
nanloss ignored
-0.7627909
nanloss ignored
nanloss ignored
nanloss ignored
-0.7870402
-0.84686506
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.76710016
-0.6334753
nanloss ignored
nanloss ignored
-0.81434655
nanloss ignored
nanloss ignored
-0.61031383
-0.60930336
nanloss ignored
-0.56168
nanloss ignored
-0.7343057
nanloss ignored
-0.7326597
nanloss ignored
-0.73240644
nanloss ignored
-0.7602447
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.7092987
nanloss ignored
-0.78076
nanloss ignored
-0.7078935
nanloss ignored
-0.728087
nanloss ignored
-0.6903667
-0.52604824
nanloss ignored
nanloss ignored
-0.67188174
-0.8327725
nanloss ignored
nanloss ignored
nanloss ignored
-0.65589446
nanloss ignored
nanloss ignored
-0.69599915
-0.89223564
-0.64483464
-0.5860321
-0.70893776
-0.78552836
nanl

nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.6161972
-0.76491976
-0.6407476
-0.50075454
nanloss ignored
nanloss ignored
-0.81067574
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.81170386
nanloss ignored
nanloss ignored
nanloss ignored
-0.7449671
-0.8611719
nanloss ignored
nanloss ignored
-0.76552576
-0.8249811
nanloss ignored
-0.7414784
-0.72653794
nanloss ignored
nanloss ignored
-0.7975004
-0.6093008
nanloss ignored
nanloss ignored
-0.6988421
-0.5699523
-0.78186804
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.5004473
-0.8131801
nanloss ignored
-0.795175
-0.83991915
-0.7933794
-0.89440215
nanloss ignored
nanloss ignored
nanloss ignored
-0.8209818
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.88563466
-0.703485
nanloss ignored
nanloss ignored
-0.70614934
-0.8376122
0.45227321237993595 1.4442796 937
0.4787892376681614 1.388717 446
0.45227321237993595 0.4787892376681614


In [57]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "verbose": True,
        "loss": "ApproxNDCGLoss",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'verbose': True, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-1.0093532e-03, -1.2634217e-03, -1.5353464e-03, ...,
         1.3662458e-03, -1.6835898e-04, -1.2866292e-03],
       [-1.6034048e-03,  1.1652935e-03, -1.0823153e-03, ...,
         1.3063130e-03, -2.1162032e-04, -4.1037455e-04],
       [ 1.2385610e-03,  1.2230030e-03,  1.2572601e-03, ...,
        -1.3639654e-03,  2.6516378e-04,  5.3027138e-04],
       ...,
       [-1.5935206e-03, -1.4302734e-03, -9.2664815e-04, ...,
        -1.3587796e-03,  9.7224233e-04,  3.5686209e-04],
       [-1.5469772e-03,  1.2565997e-03, -3.2779426e-05, ...

-0.79494685
nanloss ignored
nanloss ignored
-0.7877429
-0.8753288
nanloss ignored
nanloss ignored
-0.7638493
nanloss ignored
-0.83521163
-0.90134037
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.86541843
-0.8106544
nanloss ignored
-0.7263467
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.90593827
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8953778
nanloss ignored
-0.81484336
nanloss ignored
nanloss ignored
-0.934796
nanloss ignored
nanloss ignored
-0.6903063
-0.77574635
-0.87513477
nanloss ignored
-0.8073201
nanloss ignored
-0.8379523
-0.8093381
nanloss ignored
-0.6940891
-0.8853388
-0.87474287
nanloss ignored
nanloss ignored
-0.70967627
-0.8111256
-0.83496946
-0.90085626
nanloss ignored
-0.79701287
-0.70682746
nanloss ignored
nanloss ignored
-0.8702527
-0.8419097
nanloss ignored
-0.8641176
-0.8684921
-0.7594983
nanloss ignored


In [58]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "verbose": True,
        "loss": "qmse",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'verbose': True, 'loss': 'qmse', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 1.7194494e-03, -3.2429039e-04, -1.4837326e-03, ...,
        -4.2169823e-04,  7.5002032e-04,  7.6056796e-04],
       [ 6.6898850e-04,  1.0781651e-03, -1.4607555e-03, ...,
        -3.5210326e-04, -6.1527162e-04,  3.9947973e-04],
       [-7.0547010e-04,  2.4091273e-04,  1.2951955e-03, ...,
        -2.0558953e-04, -1.4878064e-03, -1.2623459e-03],
       ...,
       [ 5.9513358e-04,  1.5390423e-03, -2.8168410e-05, ...,
         9.4851642e-04,  1.2140283e-03,  6.5850525e-04],
       [ 1.2909532e-04, -1.1163212e-03,  4.3122427e-04, ...,
        

In [62]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "verbose": True,
        "loss": "mse",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'verbose': True, 'loss': 'mse', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 3.5321282e-04,  7.5882672e-05,  3.4894617e-04, ...,
        -1.3421698e-03,  1.4587163e-03,  2.9902012e-04],
       [ 1.8070251e-04,  4.1377038e-04,  8.4859428e-05, ...,
         2.8670504e-04, -1.6063303e-04,  1.6858331e-03],
       [ 7.2749867e-04, -5.4320716e-04, -1.6350364e-03, ...,
         8.7036489e-05, -1.4286018e-03,  9.9253480e-04],
       ...,
       [-5.0736405e-04, -8.4811560e-04,  3.1120717e-04, ...,
         5.5340421e-04,  1.2459636e-03, -5.4411771e-04],
       [-8.9793181e-04, -2.0528809e-04,  1.2387643e-03, ...,
        -

In [63]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "train_bias": True,
        "verbose": True,
        "loss": "mse",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': True, 'loss': 'mse', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-6.3450937e-04,  1.0511258e-03, -5.2001362e-04, ...,
         8.7487668e-04, -2.4695127e-04,  5.1273615e-04],
       [ 1.3468682e-03, -4.6465098e-05, -7.7752257e-04, ...,
        -5.1524938e-05, -3.7380590e-04,  3.8575724e-04],
       [-1.2716346e-03, -1.6379211e-03, -1.1778802e-03, ...,
        -1.7206289e-03,  7.5454876e-04,  7.2107482e-04],
       ...,
       [ 3.1094789e-04, -6.7457044e-04, -1.0310903e-04, ...,
         1.6503087e-03,  5.0083699e-04, -5.2996620e-04],
       [-1.2712009e-03,  4.1138887e-04,  1.2651354

In [64]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "train_bias": True,
        "verbose": True,
        "loss": "qmse",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': True, 'loss': 'qmse', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-9.0698151e-05, -4.4593483e-04, -1.0814085e-03, ...,
        -1.8825084e-04, -1.4395012e-03, -5.1791337e-04],
       [ 3.3401250e-04, -1.4738073e-03, -1.3240937e-03, ...,
         1.3445511e-03,  2.5204392e-04, -1.0266299e-03],
       [-1.4239663e-03,  1.0637313e-03,  2.1664306e-04, ...,
        -7.4056821e-04,  1.5266067e-03, -5.9437036e-04],
       ...,
       [ 3.0911565e-04,  1.9071490e-04,  1.2872779e-03, ...,
         1.1673617e-03,  2.3568720e-04, -2.8906719e-04],
       [-1.2154877e-03,  6.4180506e-04, -1.605886

In [65]:
models = [
    CBKnnV0(ctx, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "train_bias": True,
        "verbose": True,
        "loss": "qmse",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': True, 'loss': 'qmse', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.05968455e-03, -1.17955392e-03,  3.56769859e-04, ...,
         1.26745040e-03,  1.50743721e-03,  1.06452254e-03],
       [-1.59071758e-03, -1.30739179e-03, -3.24463093e-04, ...,
         1.65609631e-03, -1.11864705e-03,  4.48408719e-05],
       [-2.99185805e-04,  9.65527317e-04,  1.53890485e-03, ...,
         1.66114897e-03, -1.17253920e-03, -1.45117729e-03],
       ...,
       [-1.66360021e-03, -2.66914372e-04, -8.13357008e-04, ...,
         1.64094474e-03,  1.02675855e-04, -1.81485259e-04],
       [-6.66199427e-04,  2.8

In [66]:
models = [
    CBKnnV0(ctx, fit_kwargs={
        "train_dssm": False,
        "train_popbias": True,
        "train_bias": True,
        "verbose": True,
        "loss": "ApproxNDCGLoss",
        "loss_batch": 128,
        "c": 0.1
    }),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'train_dssm': False, 'train_popbias': True, 'train_bias': True, 'verbose': True, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'c': 0.1})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.1869298e-03,  1.2102908e-03, -4.7622769e-04, ...,
         7.3629944e-04,  7.8985211e-04,  3.6557109e-04],
       [-4.4462815e-04, -8.1440329e-04,  1.3765454e-04, ...,
         1.5874780e-03, -1.3308867e-03,  7.8529539e-04],
       [ 1.4757520e-03,  3.0132325e-04,  5.3870259e-04, ...,
        -1.0467230e-03, -1.4617629e-03, -6.4170762e-04],
       ...,
       [-4.0609704e-04,  5.9406325e-04, -2.0976439e-04, ...,
        -1.3500766e-03, -1.0090455e-03,  2.2798329e-05],
       [-1.2539659e-03, -7.4739550e-04, -7.

nanloss ignored
nanloss ignored
-0.8449718
nanloss ignored
nanloss ignored
nanloss ignored
-0.86656964
nanloss ignored
nanloss ignored
nanloss ignored
-0.8929198
nanloss ignored
-0.93932563
-0.8769741
-0.8356405
nanloss ignored
nanloss ignored
-0.88022864
-0.92875814
nanloss ignored
-0.8614006
-0.9028322
-0.9093953
nanloss ignored
-0.91294175
nanloss ignored
nanloss ignored
-0.8662158
nanloss ignored
nanloss ignored
-0.8335578
nanloss ignored
-0.81576455
nanloss ignored
-0.8433243
-0.86639005
nanloss ignored
nanloss ignored
nanloss ignored
-0.82834727
nanloss ignored
-0.80896676
-0.78208584
nanloss ignored
-0.83888227
-0.9031015
-0.8388743
nanloss ignored
-0.8472896
nanloss ignored
-0.8604877
-0.86825764
-0.88892883
nanloss ignored
-0.9226486
-0.91044164
nanloss ignored
-0.87449354
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8619263
-0.9052242
nanloss ignored
-0.90074587
nanloss ignored
-0.91460884
nanloss ignored
nanloss ignored
nanloss ignored
-

nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8653506
nanloss ignored
nanloss ignored
-0.87860894
nanloss ignored
nanloss ignored
-0.9078973
nanloss ignored
-0.8777581
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.93108636
-0.8386535
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.81228006
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8444864
nanloss ignored
nanloss ignored
nanloss ignored
-0.8706329
nanloss ignored
-0.86678696
nanloss ignored
-0.8938032
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.9340063
nanloss ignored
nanloss ignored
-0.81700164
-0.843078
-0.82279056
-0.9246554
nanloss ignored
nanloss ignored
nanloss ignored
-0.811012
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.893827

nanloss ignored
nanloss ignored
-0.87793565
nanloss ignored
nanloss ignored
-0.86610305
-0.8356167
nanloss ignored
nanloss ignored
nanloss ignored
-0.89098465
nanloss ignored
nanloss ignored
-0.9109772
-0.88577783
nanloss ignored
nanloss ignored
-0.8391592
-0.8241445
-0.8066195
-0.9277529
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.84772086
nanloss ignored
-0.9255677
-0.91687614
-0.86066
-0.8778049
nanloss ignored
nanloss ignored
nanloss ignored
-0.8821294
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8854665
nanloss ignored
nanloss ignored
-0.8380966
nanloss ignored
nanloss ignored
nanloss ignored
-0.8373234
nanloss ignored
-0.9177443
-0.85511595
nanloss ignored
nanloss ignored
nanloss ignored
-0.9032637
nanloss ignored
-0.9077026
nanloss ignored
-0.8895887
-0.82144094
-0.8365582
-0.86898637
nanloss ignored
nanloss ignored
nanloss ignored
-0.92603

-0.9142144
nanloss ignored
-0.9041379
-0.86672974
nanloss ignored
-0.86759996
-0.81267613
-0.9520259
-0.8412534
-0.8093579
nanloss ignored
nanloss ignored
-0.89074135
nanloss ignored
nanloss ignored
nanloss ignored
-0.8500326
nanloss ignored
-0.88329697
-0.8570895
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8644889
-0.83327687
-0.8218046
-0.8768136
-0.8810785
nanloss ignored
nanloss ignored
-0.8154378
-0.93174696
nanloss ignored
-0.941049
nanloss ignored
-0.8758751
-0.9374379
-0.90182644
nanloss ignored
nanloss ignored
-0.8439954
-0.82776237
-0.8543504
-0.8856336
-0.88906157
nanloss ignored
-0.89918387
nanloss ignored
nanloss ignored
nanloss ignored
-0.85158813
-0.88386667
nanloss ignored
nanloss ignored
-0.8724381
nanloss ignored
-0.9192816
-0.8335536
-0.9090063
nanloss ignored
-0.9114952
nanloss ignored
-0.8887866
-0.8465806
-0.9195028
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored

-0.8205764
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.9017228
nanloss ignored
nanloss ignored
-0.9223122
nanloss ignored
nanloss ignored
-0.89129716
-0.828919
nanloss ignored
nanloss ignored
-0.92851347
-0.8886042
-0.8281515
-0.8606437
nanloss ignored
-0.83369887
nanloss ignored
nanloss ignored
-0.8738923
-0.9037683
nanloss ignored
-0.9078014
-0.86136454
-0.8713832
-0.8970283
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.89354706
nanloss ignored
nanloss ignored
-0.83840954
-0.90622133
-0.9341621
-0.88074404
-0.91114396
nanloss ignored
-0.8865514
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.85657316
nanloss ignored
nanloss ignored
-0.8380297
-0.8838
-0.92075014
nanloss ignored
nanloss ignored
-0.94661385
-0.80389893
-0.84616214
-0.89198524
-0.8423819
-0.8826504
nanloss ignored
-0.8615119
-0.84983474
nanloss ignored
nanloss ignored
-0.85158306
-0.8329113
nanloss ignored
-0.9029774
-0.90398836
-0.871732
nan

-0.8988197
-0.87104976
-0.842361
nanloss ignored
-0.8579482
nanloss ignored
nanloss ignored
-0.86355203
-0.8797237
-0.9097861
nanloss ignored
-0.90187156
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8554797
nanloss ignored
nanloss ignored
nanloss ignored
-0.957914
-0.8330358
-0.8860354
nanloss ignored
-0.80085874
nanloss ignored
-0.85057205
-0.86089635
-0.9144013
-0.891309
-0.8668978
-0.8753456
nanloss ignored
nanloss ignored
nanloss ignored
-0.8664
nanloss ignored
nanloss ignored
nanloss ignored
-0.9293703
nanloss ignored
nanloss ignored
-0.8372971
nanloss ignored
nanloss ignored
nanloss ignored
-0.8434487
-0.83342516
nanloss ignored
-0.9248435
nanloss ignored
-0.8608017
nanloss ignored
nanloss ignored
-0.8526442
nanloss ignored
-0.8538079
-0.9344435
-0.8560642
-0.8261738
nanloss ignored
nanloss ignored
-0.9381402
-0.86425555
nanloss ignored
nanloss ignored
nanloss ignored
-0.8878649
-0.91123956
nanloss ignored
-0.86392736
nanloss ignored
-0.8234511
-0.90467066
n

nanloss ignored
-0.87733495
-0.82191175
-0.8632325
-0.9135588
nanloss ignored
nanloss ignored
-0.91744787
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.9083494
-0.9150224
-0.8994203
nanloss ignored
-0.9084282
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8137968
-0.8911597
-0.88348854
nanloss ignored
-0.9250443
nanloss ignored
nanloss ignored
nanloss ignored
-0.8632777
-0.9279025
nanloss ignored
-0.90282774
-0.8846784
nanloss ignored
-0.85268164
nanloss ignored
-0.84646845
-0.87105656
-0.84214145
nanloss ignored
nanloss ignored
nanloss ignored
-0.8751871
-0.88373667
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.89156103
nanloss ignored
nanloss ignored
nanloss ignored
-0.8639495
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
-0.8758603
-0.87185794
-0.82430065
nanloss ignored
nanloss ignored
-0.88924414
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ignored
nanloss ign

# Fit

In [42]:
models = [
    Popular(ctx),
    CBKnnV0(ctx, fit_kwargs={"c": 0.1, "train_popbias": True, "verbose": False}),
    DssmKnn(ctx, 6, fit_kwargs={"train_popbias": True, "verbose": False}),
    DssmKnn(ctx, 7, fit_kwargs={"train_popbias": True, "verbose": False}),
    DssmKnn(ctx, 8, fit_kwargs={"train_popbias": True, "verbose": False}),
    DssmKnn(ctx, 9, fit_kwargs={"train_popbias": True, "verbose": False}),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.embed_users['train'].shape 

In [69]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={"train_dssm": True, "train_popbias": True, "verbose": True, "loss": "qmse"}),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': True, 'train_popbias': True, 'verbose': True, 'loss': 'qmse'})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-7.7135919e-04, -4.9192132e-04, -8.3741394e-04, ...,
        -6.3243852e-04,  1.3664407e-03,  2.8008461e-04],
       [ 2.7533725e-04,  1.3833470e-03, -1.5620892e-03, ...,
        -1.6952583e-03,  1.0233414e-03,  4.2732432e-04],
       [ 1.6262558e-03, -1.3642908e-03, -5.4258568e-04, ...,
         8.2669826e-04,  1.7281965e-04,  1.4953753e-03],
       ...,
       [ 6.6005602e-04,  4.3044076e-04,  1.5224973e-03, ...,
        -9.0779719e-04,  6.2201096e-04, -1.0486391e-03],
       [ 1.5254053e-03, -1.7178601e-03, -1.3106528e-03, ...,
        -1.5783046e-03,  1.0775694e-03

In [70]:
models = [
    DssmKnn(ctx, 6, fit_kwargs={"train_dssm": True, "train_popbias": True, "verbose": True, "loss": "mse"}),
]

ev(models)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'train_dssm': True, 'train_popbias': True, 'verbose': True, 'loss': 'mse'})
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-2.3977309e-05, -8.7627565e-04,  8.1956742e-04, ...,
        -5.2271189e-05,  9.5829100e-04, -4.4083729e-04],
       [-8.0577098e-04,  1.2379226e-03,  8.6530385e-04, ...,
         6.0995222e-05, -1.3532357e-03,  8.2710234e-04],
       [ 1.2799114e-04, -1.1597238e-03, -6.2860339e-04, ...,
         1.2650862e-03, -8.0803974e-04, -1.6449635e-03],
       ...,
       [ 7.7366829e-05,  1.3879329e-03,  2.6216119e-04, ...,
         1.3897115e-03,  4.6908855e-07, -1.8449918e-04],
       [ 5.2927044e-04,  1.5364415e-03,  1.5827608e-03, ...,
        -2.3273780e-04,  1.2128525e-03,

In [13]:
models[-1].get_score("test"), models

0.4814798206278027 1.3964673 446


(0.4814798206278027,
 [DssmKnn(6,{'train_dssm': True, 'train_popbias': True, 'verbose': True, 'loss': 'ApproxNDCGLoss'})])

In [78]:
ck = CBKnnV0(ctx)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)


In [79]:
ck.fit(c = 1000)

self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[ 8.6122839e-04,  1.2007814e-03, -1.1980118e-03, ...,
        -1.5025315e-03,  9.2426362e-04, -1.1554204e-03],
       [-1.1495237e-03, -7.8715828e-05, -9.8494743e-04, ...,
        -9.4909963e-05,  1.3410971e-03,  4.4951020e-04],
       [-5.3058809e-04, -1.1485628e-03, -1.3411201e-03, ...,
        -9.3532167e-04,  1.3477817e-03, -3.7984966e-04],
       ...,
       [-9.6586836e-04,  1.6719025e-03, -3.4315960e-04, ...,
         3.7246198e-05, -1.2529581e-03, -1.2702995e-03],
       [ 1.0615035e-03,  1.5431535e-03, -1.2142296e-03, ...,
         1.0580012e-03,  1.3855413e-04, -6.0245988e-04],
       [ 1.3726226e-03,  1.1444500e-03,  5.9989066e-04, ...,
        -2.7345450e-04, -1.4423464e-03,  1.3075263e-04]], dtype=float32)>
0-loss =  tf.Tensor(1.6263173771745756, shape=(), dtype=float64)
1.6275129
1.5927579
1.5951067
1.5937451
1.585182
1.5784299
1.5770121
1.5774144
1.5757042
1.5716239
1.5676161
1.5659667
1.56

In [80]:
ck.get_score("train"), ck.get_score("test")

0.5495837780149413 1.5305086 937
0.5604035874439462 1.9514735 446


(0.5495837780149413, 0.5604035874439462)

In [81]:
ck = CBKnnV0(ctx)
ck.fit(c = 0.1, train_popbias=True)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-0.00022751, -0.00159857,  0.00037515, ..., -0.00016348,
        -0.00057465,  0.00132992],
       [ 0.00115122, -0.00045698,  0.00011994, ..., -0.00023302,
         0.00119457,  0.00020259],
       [ 0.00074738, -0.00028835, -0.00102171, ..., -0.00093429,
        -0.00089398,  0.00039003],
       ...,
       [ 0.00121838, -0.00088023,  0.00164141, ..., -0.00119158,
         0.00107003, -0.00054156],
       [-0.00061707,  0.00058861,  0.00139445, ...,  0.00057154,
         0.00028411,  0.00059844],
       [ 0.000497  , -0.00085014, -0.00160813, ..., -0.00035489,
         0.00072236, -0.00169149]], dtype=float32)>
0-loss =  tf.Tensor(1.6263173771745756, shape=(), dtyp

In [82]:
ck.get_score("test")

0.6008744394618835 2.1161892 446


0.6008744394618835

In [83]:
DEFAULT_FIT_KWARGS

{'learning_rate': 0.001,
 'n': 500,
 'c': 5000,
 'train_popbias': False,
 'verbose': True,
 'train_dssm': False}

In [15]:
ck = CBKnnV0(ctx, fit_kwargs={"c": 0.1, "train_popbias": True})
ck.fit(verbose = False)
ck.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.6006726457399103 2.1178312 446


0.6006726457399103

In [16]:
DEFAULT_FIT_KWARGS

{'learning_rate': 0.001,
 'n': 500,
 'c': 5000,
 'train_popbias': False,
 'verbose': True}

In [18]:
d = DssmKnn(ctx, 6, fit_kwargs={"c": 0.1, "train_popbias": True})
d.fit(verbose=False)
d.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5345067264573992 1.4824542 446


0.5345067264573992

In [96]:
d = DssmKnn(ctx, 9)
d.fit(n = 1, c = 1, train_popbias=True, verbose=False)
d.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.4719506726457399 1.5103382 446


0.4719506726457399

In [97]:
for i in range(10):
    d.W = np.eye(*d.W.shape) * i
    d.pb = 1.
    d.get_score("train")

0.4597118463180363 1.5713968168959522 937
0.47233724653148346 1.5844084901945708 937
0.48567769477054434 1.6387898525467917 937
0.48903948772678757 1.73454090395262 937
0.47970117395944506 1.8716616444120595 937
0.45564567769477055 2.050152073925099 937
0.4225400213447172 2.270012192491749 937
0.3875880469583778 2.5312420001119955 937
0.3509178228388473 2.8338414967858627 937
0.31606189967982923 3.1778106825133277 937


In [98]:
for i in range(10):
    d.W = np.eye(*d.W.shape) * i
    d.pb = 1.
    d.get_score("test")

0.47199551569506726 1.5107071184973542 446
0.4852690582959641 1.5230128244138608 446
0.4992376681614349 1.5748999921448938 446
0.5023766816143498 1.6663686216904579 446
0.4941255605381166 1.7974187130505512 446
0.4728699551569506 1.9680502662251744 446
0.440762331838565 2.1782632812143303 446
0.4025112107623319 2.428057758018013 446
0.3645739910313901 2.717433696636223 446
0.3280044843049328 3.046391097068967 446


# нужны ранжирующие лоссы!

In [27]:
score = dict()
for c in [0, .1, 1, 10, 100, 1000]:
    for m in [6,7,8,9]:
        d = DssmKnn(ctx, m)
        d.fit(c = c, train_popbias=True, verbose=False)
        score_train_ = d.get_score("train")
        score_test_ = d.get_score("test")
        print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
        score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5286125933831377 1.5397363 937
0.5377578475336323 1.4808576 446
c, m =  0 6  score_train_ =  0.5286125933831377  score_test_ =  0.5377578475336323
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5227641408751335 1.5394075 937
0.5330717488789237 1.481133 446
c, m =  0 7  score_train_ =  0.5227641408751335  score_test_ =  0.5330717488789237
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_use

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.45973319103521876 1.5710654 937
0.47204035874439454 1.5103939 446
c, m =  1000 7  score_train_ =  0.45973319103521876  score_test_ =  0.47204035874439454
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.4611632870864461 1.5708507 937
0.4731390134529148 1.5101732 446
c, m =  1000 8  score_train_ =  0.4611632870864461  score_test_ =  0.4731390134529148
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)


In [28]:
score

{(0, 6): 0.5377578475336323,
 (0, 7): 0.5330717488789237,
 (0, 8): 0.5355381165919283,
 (0, 9): 0.48950672645739907,
 (0.1, 6): 0.5342825112107624,
 (0.1, 7): 0.5288789237668161,
 (0.1, 8): 0.5335874439461883,
 (0.1, 9): 0.4916143497757847,
 (1, 6): 0.5181838565022422,
 (1, 7): 0.509439461883408,
 (1, 8): 0.5195964125560538,
 (1, 9): 0.4912780269058296,
 (10, 6): 0.49024663677130037,
 (10, 7): 0.48213004484304933,
 (10, 8): 0.4872645739910314,
 (10, 9): 0.47757847533632286,
 (100, 6): 0.4734977578475337,
 (100, 7): 0.47298206278026905,
 (100, 8): 0.47390134529147987,
 (100, 9): 0.4730044843049327,
 (1000, 6): 0.47199551569506726,
 (1000, 7): 0.47204035874439454,
 (1000, 8): 0.4731390134529148,
 (1000, 9): 0.47199551569506726}

In [26]:
for c in [0, .1, 1, 10, 100, 1000]:
    d = CBKnnV0(ctx)
    d.fit(c = c, train_popbias=True, verbose=False)
    score_train_ = d.get_score("train")
    score_test_ = d.get_score("test")
    print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
    score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5949839914621131 1.4684364 937
0.6007847533632287 2.1153526 446
c, m =  0 8  score_train_ =  0.5949839914621131  score_test_ =  0.6007847533632287
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5953255069370331 1.4684305 937
0.6005829596412556 2.1192644 446
c, m =  0.1 8  score_train_ =  0.5953255069370331  score_test_ =  0.6005829596412556
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_

In [85]:
score = dict()
for m in [6,7,8,9]:
    d = DssmKnn(ctx, m)
    d.fit(c = c, train_dssm=True, verbose = False)
    score_train_ = d.get_score("train")
    score_test_ = d.get_score("test")
    print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
    score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.14362860192102456 1.6231251 937
0.14405829596412556 1.56129 446
c, m =  1000 6  score_train_ =  0.14362860192102456  score_test_ =  0.14405829596412556
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.10878335112059766 1.6229208 937
0.11165919282511211 1.561061 446
c, m =  1000 7  score_train_ =  0.10878335112059766  score_test_ =  0.11165919282511211
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)

In [87]:
score = dict()
for m in [6,7,8,9]:
    d = DssmKnn(ctx, m)
    d.fit(c = c, train_dssm=True, train_popbias=True, verbose = False)
    score_train_ = d.get_score("train")
    score_test_ = d.get_score("test")
    print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
    score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.4597118463180363 1.5709366 937
0.47199551569506726 1.5102675 446
c, m =  1000 6  score_train_ =  0.4597118463180363  score_test_ =  0.47199551569506726
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.45973319103521876 1.571064 937
0.4721300448430493 1.5103929 446
c, m =  1000 7  score_train_ =  0.45973319103521876  score_test_ =  0.4721300448430493
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
s

In [88]:
d = CBKnnV0(ctx)
d.fit(c = c, train_dssm=True, train_popbias=False, verbose=False)
score_train_ = d.get_score("train")
score_test_ = d.get_score("test")
print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5857097118463179 1.34445 937
0.5787892376681614 3.1525443 446
c, m =  1000 9  score_train_ =  0.5857097118463179  score_test_ =  0.5787892376681614


In [89]:
d = CBKnnV0(ctx)
d.fit(c = c, train_dssm=True, train_popbias=True, verbose=False)
score_train_ = d.get_score("train")
score_test_ = d.get_score("test")
print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5696264674493063 1.3766221 937
0.5716143497757847 2.5482621 446
c, m =  1000 9  score_train_ =  0.5696264674493063  score_test_ =  0.5716143497757847


# EOLN

In [127]:
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from sklearn.metrics import pairwise 

In [ ]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

In [23]:
import matplotlib.pyplot as plt
plt.hist(embed_games.flatten())

(array([1.667877e+06, 2.000000e+01, 6.000000e+00, 4.000000e+00,
        1.000000e+00, 4.000000e+00, 0.000000e+00, 0.000000e+00,
        1.000000e+00, 1.000000e+00]),
 array([9.74706709e-05, 6.56539696e+01, 1.31307842e+02, 1.96961714e+02,
        2.62615586e+02, 3.28269458e+02, 3.93923330e+02, 4.59577202e+02,
        5.25231074e+02, 5.90884946e+02, 6.56538818e+02]),
 <a list of 10 Patch objects>)

In [25]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.005857787810383747 0.4473702031602709
d -> 0.4814559819413093 0.006038374717832957 0.4473702031602709
c -> 0.01746049661399549 0.005959367945823928 0.4473702031602709
w -> 0.4944808126410835 0.0060835214446952595 0.4473702031602709


In [26]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.003182844243792325 0.3336343115124153
d -> 0.36880361173814896 0.0031828442437923255 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.38158013544018066 0.003069977426636569 0.3336343115124153


Тюним

In [135]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 1000
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)
1.3766173
1.2021382
1.1933919
1.1919646
1.1917517
1.1917268
1.1917244
1.1917244
1.1917244
1.1917243


In [28]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.002618510158013544 0.3336343115124153
d -> 0.36880361173814896 0.0029119638826185104 0.3336343115124153
c -> 0.008577878103837472 0.002957110609480813 0.3336343115124153
w -> 0.40388261851015805 0.0027539503386004517 0.3336343115124153


In [29]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.006422121896162527 0.4473702031602709
d -> 0.4814559819413093 0.0060609480812641075 0.4473702031602709
c -> 0.01746049661399549 0.005711060948081265 0.4473702031602709
w -> 0.512020316027088 0.005970654627539504 0.4473702031602709


# dssm?

In [136]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-docblocks[i] @ requests_i[1][i] for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.15483069977426636 0.0031602708803611735 0.3336343115124153
7 -> 0.11663656884875846 0.0034085778781038373 0.3336343115124153
8 -> 0.15386004514672688 0.0032054176072234763 0.3336343115124153
9 -> 0.0362979683972912 0.0032054176072234763 0.3336343115124153
e -> 0.3762076749435666 0.0031602708803611735 0.3336343115124153
d -> 0.36880361173814896 0.0028442437923250565 0.3336343115124153
c -> 0.008577878103837472 0.0029119638826185104 0.3336343115124153
w -> 0.40388261851015805 0.002618510158013544 0.3336343115124153


In [137]:
games_top

array([0.0047713 , 0.00415357, 0.00380251, ..., 0.1423562 , 0.04750573,
       0.07614477])

In [141]:
dssmid = 7
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
7 -> 0.3336343115124153 0.0031602708803611735 0.3336343115124153
x =  1
7 -> 0.35379232505643343 0.0027765237020316025 0.3336343115124153
x =  2
7 -> 0.37090293453724604 0.0031602708803611735 0.3336343115124153
x =  3
7 -> 0.38564334085778784 0.002889390519187359 0.3336343115124153
x =  4
7 -> 0.40259593679458244 0.002663656884875847 0.3336343115124153
x =  5
7 -> 0.4182844243792325 0.002641083521444696 0.3336343115124153
x =  6
7 -> 0.4323702031602709 0.0031602708803611743 0.3336343115124153
x =  7
7 -> 0.4396839729119639 0.0028668171557562076 0.3336343115124153
x =  8
7 -> 0.44162528216704294 0.0034085778781038373 0.3336343115124153
x =  9
7 -> 0.4411286681715576 0.0035665914221218965 0.3336343115124153
x =  10
7 -> 0.4392099322799097 0.0030248306997742664 0.3336343115124153
x =  11
7 -> 0.43462753950338606 0.0030248306997742664 0.3336343115124153
x =  12
7 -> 0.4294356659142212 0.003386004514672686 0.3336343115124153
x =  13
7 -> 0.42151241534988715 0.003363431151241535 0.333

In [142]:
dssmid = 6
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.3336343115124153 0.0030925507900677203 0.3336343115124153
x =  1
6 -> 0.37266365688487585 0.003002257336343115 0.3336343115124153
x =  2
6 -> 0.3972911963882618 0.003340857787810384 0.3336343115124153
x =  3
6 -> 0.42458239277652365 0.0024830699774266367 0.3336343115124153
x =  4
6 -> 0.45449209932279916 0.003115124153498871 0.3336343115124153
x =  5
6 -> 0.4766591422121897 0.0030699774266365687 0.3336343115124153
x =  6
6 -> 0.4895033860045147 0.0032054176072234763 0.3336343115124153
x =  7
6 -> 0.4941083521444696 0.0032731376975169302 0.3336343115124153
x =  8
6 -> 0.49146726862302487 0.003002257336343115 0.3336343115124153
x =  9
6 -> 0.48632054176072237 0.002595936794582393 0.3336343115124153
x =  10
6 -> 0.4764559819413092 0.0028893905191873593 0.3336343115124153
x =  11
6 -> 0.4654853273137698 0.0027313769751693 0.3336343115124153
x =  12
6 -> 0.45309255079006777 0.0030248306997742672 0.3336343115124153
x =  13
6 -> 0.43932279909706545 0.003024830699774266 0.3336343

In [152]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.3336343115124153 0.0029345372460496616 0.3336343115124153
x =  1
9 -> 0.34738148984198647 0.002844243792325057 0.3336343115124153
x =  2
9 -> 0.3593227990970655 0.003002257336343115 0.3336343115124153
x =  3
9 -> 0.3699097065462754 0.0030248306997742672 0.3336343115124153
x =  4
9 -> 0.37024830699774264 0.003024830699774266 0.3336343115124153
x =  5
9 -> 0.3647178329571106 0.002595936794582393 0.3336343115124153
x =  6
9 -> 0.3520767494356659 0.0028668171557562076 0.3336343115124153
x =  7
9 -> 0.3291196388261851 0.003047404063205418 0.3336343115124153
x =  8
9 -> 0.30002257336343113 0.002641083521444696 0.3336343115124153
x =  9
9 -> 0.2689390519187359 0.002889390519187359 0.3336343115124153
x =  10
9 -> 0.24376975169300227 0.0027313769751693 0.3336343115124153
x =  11
9 -> 0.22112866817155757 0.0031828442437923255 0.3336343115124153
x =  12
9 -> 0.20067720090293456 0.0034085778781038373 0.3336343115124153
x =  13
9 -> 0.18354401805869075 0.0030925507900677203 0.33363431

In [143]:
for x in range(20):
    print("x = ", x)
    rank_w = np.argsort(-x*embed_users @ W @ embed_games.T-games_top, axis=1)


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_w, "w"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
w -> 0.3336343115124153 0.0032279909706546274 0.3336343115124153
x =  1
w -> 0.39647855530474035 0.0033182844243792326 0.3336343115124153
x =  2
w -> 0.40907449209932284 0.0030925507900677203 0.3336343115124153
x =  3
w -> 0.4110835214446953 0.0031828442437923255 0.3336343115124153
x =  4
w -> 0.41049661399548537 0.0035214446952595937 0.3336343115124153
x =  5
w -> 0.4100451467268624 0.0034085778781038373 0.3336343115124153
x =  6
w -> 0.4096839729119639 0.003340857787810384 0.3336343115124153
x =  7
w -> 0.40936794582392777 0.0034762979683972913 0.3336343115124153
x =  8
w -> 0.40880361173814905 0.0031602708803611735 0.3336343115124153
x =  9
w -> 0.4087810383747179 0.0030248306997742664 0.3336343115124153
x =  10
w -> 0.4089164785553048 0.002934537246049662 0.3336343115124153
x =  11
w -> 0.4089164785553048 0.0027313769751693 0.3336343115124153
x =  12
w -> 0.40887133182844243 0.003295711060948081 0.3336343115124153
x =  13
w -> 0.4086004514672686 0.002595936794582393 0.333634

In [147]:
print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)


In [148]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T-games_top) ** 2)
            + tf.reduce_mean(W * W) * 150
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
1.3189981
1.1485099
1.1218199
1.110669
1.1049802
1.1020124
1.1004989
1.0997444
1.0993747
1.0991968


In [149]:
rank_w = np.argsort(-embed_users @ W @ embed_games.T-games_top, axis=1)


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_w, "w"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids

            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

w -> 0.4295936794582393 0.002595936794582393 0.3336343115124153


In [41]:
500 -> .4108
250 -> .4117
150 -> .4143
100 -> .4151
050 -> .4142
010 -> .4115
000 -> .4033

SyntaxError: invalid syntax (<ipython-input-41-1b4e77c0b5a7>, line 1)

In [150]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.4941083521444696 0.0028216704288939053 0.3336343115124153
7 -> 0.4396839729119639 0.0032054176072234763 0.3336343115124153
8 -> 0.47376975169300223 0.0033182844243792326 0.3336343115124153
9 -> 0.3291196388261851 0.0028668171557562076 0.3336343115124153
e -> 0.3762076749435666 0.0031376975169300227 0.3336343115124153
d -> 0.36880361173814896 0.0027539503386004517 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.4295936794582393 0.0032054176072234763 0.3336343115124153


In [153]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5568735891647856 0.006151241534988712 0.4473702031602709
7 -> 0.5192212189616252 0.005846501128668171 0.4473702031602709
8 -> 0.567178329571106 0.006455981941309256 0.4473702031602709
9 -> 0.37682844243792324 0.006264108352144469 0.4473702031602709
e -> 0.4731151241534989 0.006218961625282167 0.4473702031602709
d -> 0.4814559819413093 0.006060948081264108 0.4473702031602709
c -> 0.01746049661399549 0.005812641083521445 0.4473702031602709
w -> 0.527155756207675 0.006060948081264108 0.4473702031602709


In [154]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-5 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5759367945823928 0.005711060948081263 0.4473702031602709
7 -> 0.5242776523702032 0.006015801354401806 0.4473702031602709
8 -> 0.5645372460496614 0.005744920993227991 0.4473702031602709
9 -> 0.44339729119638827 0.006015801354401806 0.4473702031602709
e -> 0.4731151241534989 0.005970654627539504 0.4473702031602709
d -> 0.4814559819413093 0.006049661399548532 0.4473702031602709
c -> 0.01746049661399549 0.006309255079006772 0.4473702031602709
w -> 0.527155756207675 0.006230248306997742 0.4473702031602709


In [160]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 200
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.666410835214447 0.01167607223476298 0.5744469525959367
7 -> 0.6234255079006772 0.011834085778781037 0.5744469525959367
8 -> 0.6523137697516931 0.012178329571106095 0.5744469525959367
9 -> 0.5699266365688487 0.011811512415349886 0.5744469525959367
e -> 0.5722686230248306 0.01195823927765237 0.5744469525959367
d -> 0.5961343115124152 0.012076749435665914 0.5744469525959367
c -> 0.030299097065462757 0.012753950338600452 0.5744469525959367
w -> 0.626089164785553 0.011726862302483071 0.5744469525959367


In [161]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 250
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.6831467268623025 0.014785553047404065 0.5979503386004515
7 -> 0.6498690744920992 0.015480812641083523 0.5979503386004515
8 -> 0.6795124153498872 0.01450112866817156 0.5979503386004515
9 -> 0.5858916478555304 0.014961625282167042 0.5979503386004515
e -> 0.592334085778781 0.015431151241534989 0.5979503386004515
d -> 0.6212821670428893 0.015327313769751695 0.5979503386004515
c -> 0.03572460496613995 0.015449209932279913 0.5979503386004515
w -> 0.6479593679458239 0.015462753950338602 0.5979503386004515


In [51]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.5744469525959367 0.012121896162528218 0.5744469525959367
x =  1
6 -> 0.6088205417607224 0.01252257336343115 0.5744469525959367
x =  2
6 -> 0.6479627539503386 0.01204288939051919 0.5744469525959367
x =  3
6 -> 0.664283295711061 0.012088036117381492 0.5744469525959367
x =  4
6 -> 0.6479740406320542 0.011580135440180587 0.5744469525959367
x =  5
6 -> 0.6167832957110609 0.011839729119638827 0.5744469525959367
x =  6
6 -> 0.5822968397291196 0.012025959367945822 0.5744469525959367
x =  7
6 -> 0.5485609480812641 0.012291196388261852 0.5744469525959367
x =  8
6 -> 0.5183069977426636 0.012127539503386006 0.5744469525959367
x =  9
6 -> 0.49157449209932275 0.011930022573363432 0.5744469525959367
x =  10
6 -> 0.46870203160270885 0.012020316027088036 0.5744469525959367
x =  11
6 -> 0.44742099322799095 0.012477426636568848 0.5744469525959367
x =  12
6 -> 0.4283465011286682 0.012116252821670429 0.5744469525959367
x =  13
6 -> 0.41200338600451464 0.012003386004514673 0.5744469525959367
x

In [52]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.4473702031602709 0.0059480812641083515 0.4473702031602709
x =  1
6 -> 0.4830135440180587 0.006241534988713319 0.4473702031602709
x =  2
6 -> 0.5132167042889391 0.006478555304740407 0.4473702031602709
x =  3
6 -> 0.5453160270880361 0.006038374717832957 0.4473702031602709
x =  4
6 -> 0.56813769751693 0.006207674943566591 0.4473702031602709
x =  5
6 -> 0.5727878103837472 0.006015801354401806 0.4473702031602709
x =  6
6 -> 0.5652934537246049 0.005857787810383747 0.4473702031602709
x =  7
6 -> 0.5506546275395033 0.005575620767494356 0.4473702031602709
x =  8
6 -> 0.5308352144469526 0.005744920993227991 0.4473702031602709
x =  9
6 -> 0.5122234762979685 0.006106094808126411 0.4473702031602709
x =  10
6 -> 0.4936230248306998 0.006591422121896162 0.4473702031602709
x =  11
6 -> 0.47469525959367953 0.00618510158013544 0.4473702031602709
x =  12
6 -> 0.45573363431151237 0.006218961625282167 0.4473702031602709
x =  13
6 -> 0.43795711060948084 0.0059480812641083515 0.4473702031602709


In [158]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.4473702031602709 0.0063318284424379225 0.4473702031602709
x =  1
9 -> 0.45654627539503384 0.005846501128668171 0.4473702031602709
x =  2
9 -> 0.4679345372460497 0.006331828442437923 0.4473702031602709
x =  3
9 -> 0.4737810383747178 0.0060270880361173815 0.4473702031602709
x =  4
9 -> 0.46619638826185095 0.006218961625282167 0.4473702031602709
x =  5
9 -> 0.44339729119638827 0.006399548532731377 0.4473702031602709
x =  6
9 -> 0.4103386004514672 0.006399548532731377 0.4473702031602709
x =  7
9 -> 0.37682844243792324 0.005688487584650113 0.4473702031602709
x =  8
9 -> 0.3405079006772009 0.005654627539503386 0.4473702031602709
x =  9
9 -> 0.30753950338600455 0.0060270880361173815 0.4473702031602709
x =  10
9 -> 0.27742663656884875 0.006309255079006772 0.4473702031602709
x =  11
9 -> 0.25196388261851016 0.006162528216704289 0.4473702031602709
x =  12
9 -> 0.23016930022573365 0.005948081264108352 0.4473702031602709
x =  13
9 -> 0.2112641083521445 0.006252821670428894 0.44737020

In [159]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.5744469525959367 0.012392776523702033 0.5744469525959367
x =  1
9 -> 0.584018058690745 0.012020316027088036 0.5744469525959367
x =  2
9 -> 0.586410835214447 0.011653498871331828 0.5744469525959367
x =  3
9 -> 0.5699266365688487 0.012279909706546277 0.5744469525959367
x =  4
9 -> 0.5346388261851016 0.011822799097065465 0.5744469525959367
x =  5
9 -> 0.49005079006772007 0.012133182844243792 0.5744469525959367
x =  6
9 -> 0.44170993227990973 0.011918735891647854 0.5744469525959367
x =  7
9 -> 0.39646726862302484 0.012466139954853276 0.5744469525959367
x =  8
9 -> 0.35639954853273137 0.011952595936794583 0.5744469525959367
x =  9
9 -> 0.32224040632054174 0.011997742663656885 0.5744469525959367
x =  10
9 -> 0.2926354401805869 0.012042889390519187 0.5744469525959367
x =  11
9 -> 0.2685214446952596 0.01260722347629797 0.5744469525959367
x =  12
9 -> 0.2483803611738149 0.012641083521444697 0.5744469525959367
x =  13
9 -> 0.23149548532731376 0.012127539503386006 0.5744469525959367